# LangGraph - Hierarchical Agent Teams

> A Guide to Modular Agent Orchestration

In the previous chapter, we introduced the supervisor agent pattern, where a supervisor agent was responsible for overseeing worker agents and managing their behaviour.

![](assets/img/15-langgraph-hierarchical-01.gif)

We can now take this a step further by creating more advanced workflows in which multiple supervisor agents collaborate harmoniously within a hierarchical structure.

![](assets/img/15-langgraph-hierarchical-02.webp)

In this chapter, we explore a hierarchical system where responsibilities are distributed among specialized supervisor agents, each managing their own team of worker agents. These supervisors can route tasks between teams, coordinate outputs, and collaborate to fulfill complex user requests.

For instance, we might divide responsibilities between a Research Team Supervisor and a Document Authoring Supervisor. Each oversees a subset of agents specialized in tasks like web scraping, searching, writing, or note-taking. This modular approach allows fo greater scalability, parallelism, and clearer delegation of responsibilities across the system.

Let's start by defining a simple web scraping tool that the Research Team might use when tasked with gathering detailed content from a list of URLs.

In [1]:
from typing import List
from langchain_core.tools import tool
from langchain_community.document_loaders import WebBaseLoader

@tool
def scrape_webpages(urls: List[str]) -> str:
    """Use requests and bs4 to scrape the provided web pages for detailed information."""
    loader = WebBaseLoader(urls)
    docs = loader.load()
    return "\n\n".join(
        [
            f'<Document name="{doc.metadata.get("title", "")}">\n{doc.page_content}\n</Document>'
            for doc in docs
        ]
    )

USER_AGENT environment variable not set, consider setting it to identify your requests.


To support the Document Authoring Team, we can provide a set of tools tailored for drafting, reading, editing, and executing auxiliary code.

These tools empower agents like the Writer, Note Taker, and Chart Generator to operate collaboratively on shared files.

In [ ]:
from langchain_experimental.utilities import PythonREPL
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Dict, Optional, Annotated, TypedDict

_TEMP_DIRECTORY = TemporaryDirectory()
WORKING_DIRECTORY = Path(_TEMP_DIRECTORY.name)

@tool
def create_outline(
    points: Annotated[List[str], "List of main points or sections."],
    file_name: Annotated[str, "File path to save the outline."],
) -> Annotated[str, "Path of the saved outline file."]:
    """Create and save an outline."""
    with (WORKING_DIRECTORY / file_name).open("w") as file:
        for i, point in enumerate(points):
            file.write(f"{i + 1}. {point}\n")
    return f"Outline saved to {file_name}"


@tool
def read_document(
    file_name: Annotated[str, "File path to read the document from."],
    start: Annotated[Optional[int], "The start line. Default is 0"] = None,
    end: Annotated[Optional[int], "The end line. Default is None"] = None,
) -> str:
    """Read the specified document."""
    with (WORKING_DIRECTORY / file_name).open("r") as file:
        lines = file.readlines()
    if start is None:
        start = 0
    return "\n".join(lines[start:end])


@tool
def write_document(
    content: Annotated[str, "Text content to be written into the document."],
    file_name: Annotated[str, "File path to save the document."],
) -> Annotated[str, "Path of the saved document file."]:
    """Create and save a text document."""
    with (WORKING_DIRECTORY / file_name).open("w") as file:
        file.write(content)
    return f"Document saved to {file_name}"


@tool
def edit_document(
    file_name: Annotated[str, "Path of the document to be edited."],
    inserts: Annotated[
        Dict[int, str],
        "Dictionary where key is the line number (1-indexed) and value is the text to be inserted at that line.",
    ],
) -> Annotated[str, "Path of the edited document file."]:
    """Edit a document by inserting text at specific line numbers."""

    with (WORKING_DIRECTORY / file_name).open("r") as file:
        lines = file.readlines()

    sorted_inserts = sorted(inserts.items())

    for line_number, text in sorted_inserts:
        if 1 <= line_number <= len(lines) + 1:
            lines.insert(line_number - 1, text + "\n")
        else:
            return f"Error: Line number {line_number} is out of range."

    with (WORKING_DIRECTORY / file_name).open("w") as file:
        file.writelines(lines)

    return f"Document edited and saved to {file_name}"


# Warning: This executes code locally, which can be unsafe when not sandboxed

repl = PythonREPL()


@tool
def python_repl_tool(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use this to execute python code. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return f"Successfully executed:\n```python\n{code}\n```\nStdout: {result}"

* **`create_outline`**: Allows an agent (like the Writer) to generate a high-level document structure based on given section titles. The output is saved to a file in the working directory.

* **`read_document`**: Lets agents read either the full document or a range of lines. Useful for referencing past sections, reviewing edits, or appending content incrementally.

* **`write_document`**: Used for saving complete drafts or finalized content. This is how agents persist their contributions to disk.

* **`edit_document`**: Supports targeted edits by inserting lines at specified positions. Ideal for iterative collaboration, where agents refine or expand previous content.

* **`python_repl_tool`**: Grants agents like the Chart Generator the ability to run Python code dynamically — whether it's generating data, producing visualizations, or calculating values. The code output is returned alongside printed results.

When designing modular multi-agent systems, one of the key challenges is managing complexity as our workflow scales. To streamline the construction of these systems, we can introduce **utility functions** that simplify the creation of *worker agents* and *supervisor nodes*.

Let’s start by creating a function that encapsulates the logic for building a **supervisor node**.

In [2]:
from langchain_core.language_models.chat_models import BaseChatModel
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
from typing import Literal


class State(MessagesState):
    next: str


def make_supervisor_node(llm: BaseChatModel, members: list[str]) -> str:
    options = ["FINISH"] + members
    system_prompt = (
        "You are a supervisor tasked with managing a conversation between the"
        f" following workers: {members}. Given the following user request,"
        " respond with the worker to act next. Each worker will perform a"
        " task and respond with their results and status. When finished,"
        " respond with FINISH."
    )

    class Router(TypedDict):
        """Worker to route to next. If no workers needed, route to FINISH."""

        next: Literal[*options]

    def supervisor_node(state: State) -> Command[Literal[*members, "__end__"]]:
        """An LLM-based router."""
        messages = [
            {"role": "system", "content": system_prompt},
        ] + state["messages"]
        response = llm.with_structured_output(Router).invoke(messages)
        goto = response["next"]
        if goto == "FINISH":
            goto = END

        return Command(goto=goto, update={"next": goto})

    return supervisor_node